# ML_U3_C02 — Entrenamiento Robusto: Regularización, Optimización y PyTorch

📝 **Modalidad: Clase interactiva — sigue junto al profesor.**

**Versión:** 2025-1 | **Modificado:** 2026-05-09

---

## 📋 Mapa de la clase

| Sección | Tema | Tiempo |
|---------|------|--------|
| 1 | El problema del sobreajuste en redes neuronales | 10 min |
| 2 | Regularización: L2, Dropout y Batch Normalization | 30 min |
| 3 | Optimización: más allá del SGD básico | 25 min |
| 4 | Introducción a PyTorch: red desde cero | 30 min |
| 5 | Experimento: comparar estrategias de regularización | 10 min |
| 6 | Ejercicio en clase | 10 min |
| 7 | Resumen, conexiones y cierre de la unidad | 5 min |

---

## 📚 Prerequisitos

### 🔵 Pregrado
- Clase anterior: Perceptrón, MLP, Backpropagation
- Entrenamiento con `MLPClassifier` de sklearn
- Noción de sobreajuste/subajuste y curvas de aprendizaje

### 🟡 Doctorado
- Todo lo anterior, además:
- Derivación de backpropagation y variables adjuntas
- Optimización: gradiente descendente, convergencia
- Probabilidad bayesiana (para interpretación de regularización L2)

---

## 🎯 Objetivos de aprendizaje

Al terminar esta clase podrás:
- Aplicar Dropout y regularización L2 para controlar el sobreajuste
- Explicar qué hace Batch Normalization y cuándo usarlo
- Comparar Adam, SGD con momentum y RMSprop empíricamente
- Implementar una red neuronal completa en PyTorch con train loop propio
- **(Doctorado)** Derivar la actualización de Adam, interpretar L2 como prior gaussiano


## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.datasets import make_moons, load_digits, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Verificar disponibilidad de PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f"✅ Setup completo | PyTorch {torch.__version__} disponible")
except ImportError:
    TORCH_AVAILABLE = False
    print("✅ Setup completo | ⚠️  PyTorch no instalado — Sección 4 usará sklearn como sustituto")
    print("   Para instalar: pip install torch torchvision")

import sklearn; import numpy as np; import pandas as pd
print(f"   numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__}")

---
## Sección 1 — El Problema del Sobreajuste en Redes Neuronales

Las redes neuronales tienen millones de parámetros. Con suficientes parámetros, una red puede **memorizar** el conjunto de entrenamiento perfectamente — y generalizar muy mal.

> ⚠️ Una red que memoriza el entrenamiento tiene accuracy=1.0 en train pero puede colapsar en test.

El diagnóstico clásico es la **brecha train-validación** en las curvas de aprendizaje.


In [ ]:
# ━━━ DEMOSTRACIÓN: SOBREAJUSTE CONTROLADO ━━━
# Dataset pequeño para forzar sobreajuste

X_over, y_over = make_moons(n_samples=150, noise=0.25, random_state=RANDOM_STATE)
X_otr, X_ote, y_otr, y_ote = train_test_split(
    X_over, y_over, test_size=0.4, random_state=RANDOM_STATE
)

# Red pequeña (underfitting) vs. red grande sin regularización (overfitting)
configs = {
    'Subajuste (10,)':        MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=RANDOM_STATE),
    'Sobreajuste (200,200,200)': MLPClassifier(hidden_layer_sizes=(200,200,200), max_iter=500, random_state=RANDOM_STATE),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
h = 0.02
x0r = np.arange(X_over[:,0].min()-0.3, X_over[:,0].max()+0.3, h)
x1r = np.arange(X_over[:,1].min()-0.3, X_over[:,1].max()+0.3, h)
xx, yy = np.meshgrid(x0r, x1r)

for ax, (name, clf) in zip(axes, configs.items()):
    pipe = Pipeline([('sc', StandardScaler()), ('clf', clf)])
    pipe.fit(X_otr, y_otr)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_otr[:,0], X_otr[:,1], c=y_otr, cmap='RdBu', edgecolors='black', s=50, label='Train')
    ax.scatter(X_ote[:,0], X_ote[:,1], c=y_ote, cmap='RdBu', edgecolors='gray', s=50, marker='^', alpha=0.6, label='Test')
    tr_acc = accuracy_score(y_otr, pipe.predict(X_otr))
    te_acc = accuracy_score(y_ote, pipe.predict(X_ote))
    ax.set_title(f'{name}\nTrain: {tr_acc:.3f} | Test: {te_acc:.3f}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Subajuste vs. Sobreajuste en MLP', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("💡 La red grande memoriza el train pero su frontera es muy errática en zonas sin datos")

---
## Sección 2 — Regularización: Tres Técnicas Fundamentales

Regularización es cualquier técnica que reduce el sobreajuste induciendo restricciones en el modelo.
Veremos las tres más usadas en redes neuronales profundas.

### 2.1 — Regularización L2 (Weight Decay)

Añade una penalización proporcional a la norma cuadrática de los pesos a la función de pérdida:

$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \frac{\alpha}{2} \sum_{l,i,j} (W^{(l)}_{ij})^2$$

El efecto es empujar todos los pesos hacia cero, evitando que la red confíe en demasiados features.


In [ ]:
# ━━━ EFECTO DE alpha (L2) — BARRIDO SISTEMÁTICO ━━━
from sklearn.model_selection import cross_val_score

alphas = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0]
cv_means, cv_stds = [], []

for a in alphas:
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), alpha=a,
                              max_iter=500, random_state=RANDOM_STATE))
    ])
    scores = cross_val_score(pipe, X_over, y_over, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(alphas, cv_means, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.fill_between(alphas,
                [m-s for m,s in zip(cv_means,cv_stds)],
                [m+s for m,s in zip(cv_means,cv_stds)],
                alpha=0.15, color='steelblue')
ax.set_xlabel('alpha (escala log)'); ax.set_ylabel('Accuracy CV (5-fold)')
ax.set_title('Efecto de la Regularización L2 (alpha) — Red (100,50)')

best_i = int(np.argmax(cv_means))
ax.axvline(alphas[best_i], color='tomato', linestyle='--', alpha=0.7,
           label=f'Mejor alpha={alphas[best_i]:.0e} (acc={cv_means[best_i]:.3f})')
ax.legend()
plt.tight_layout(); plt.show()

print(f"Mejor alpha: {alphas[best_i]:.0e}  →  CV accuracy: {cv_means[best_i]:.4f}")

### 2.2 — Dropout

Durante el entrenamiento, **apaga aleatoriamente** una fracción $p$ de neuronas en cada forward pass.
En inferencia, todas las neuronas están activas pero sus pesos se escalan por $(1-p)$.

> 💡 Dropout entrena $2^n$ redes distintas (una por cada máscara posible) y las promedia implícitamente.


### 2.3 — Batch Normalization

Normaliza las activaciones de cada capa durante el entrenamiento, usando la media y
desviación estándar del mini-batch actual. Luego las reescala con parámetros aprendibles
$\gamma$ y $\beta$.

$$\hat{z}_i = \frac{z_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{z}_i + \beta$$

Efectos: permite usar learning rates más altos, actúa como regularizador, reduce sensibilidad a la inicialización.


---
## Sección 3 — Optimización: Más Allá del SGD Básico

El gradiente descendente estocástico (SGD) actualiza los pesos con el gradiente
calculado sobre un mini-batch. Sus variantes modernas aceleran la convergencia
y reducen la sensibilidad a la tasa de aprendizaje.

| Optimizador | Idea central | Ventaja | Desventaja |
|------------|-------------|---------|------------|
| SGD | Solo el gradiente actual | Simple, buen sesgo inductivo | Lento, sensible a lr |
| SGD + Momentum | Acumula velocidad en la dirección del gradiente | Supera mesetas | Sobreoscilación posible |
| RMSprop | Divide por la raíz de la varianza reciente | Adapta lr por parámetro | Sin estado de primer momento |
| Adam | Combina momentum + RMSprop | Rápido, robusto | Puede generalizar peor que SGD en algunos casos |


In [ ]:
# ━━━ COMPARACIÓN EMPÍRICA DE OPTIMIZADORES ━━━
# Simulamos diferentes optimizadores usando sklearn con parámetros equivalentes

from sklearn.model_selection import cross_val_score

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

optimizers_configs = {
    'SGD (lr=0.001)':        dict(solver='sgd', learning_rate_init=0.001, momentum=0.0),
    'SGD+Momentum (0.9)':    dict(solver='sgd', learning_rate_init=0.001, momentum=0.9),
    'Adam (lr=0.001)':       dict(solver='adam', learning_rate_init=0.001),
}

print("Comparación de optimizadores — Dataset Dígitos (5-fold CV)\n")
print(f"{'Optimizador':<25} {'CV Accuracy':>12} {'Std':>7}")
print("-" * 47)

results_opt = {}
for name, kwargs in optimizers_configs.items():
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                              random_state=RANDOM_STATE, **kwargs))
    ])
    scores = cross_val_score(pipe, X_dig, y_dig, cv=5, scoring='accuracy')
    results_opt[name] = scores
    print(f"{name:<25} {scores.mean():>12.4f} {scores.std():>7.4f}")

In [ ]:
# ━━━ LEARNING RATE SCHEDULING ━━━
# Entrenar con tasa de aprendizaje decreciente ('adaptive' en sklearn)

print("Efecto del learning rate schedule (adaptive vs. constant):\n")
for lr_schedule in ['constant', 'adaptive']:
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), solver='sgd',
                              learning_rate=lr_schedule, learning_rate_init=0.1,
                              momentum=0.9, max_iter=500, random_state=RANDOM_STATE))
    ])
    scores = cross_val_score(pipe, X_dig, y_dig, cv=5, scoring='accuracy')
    print(f"  {lr_schedule:>10}:  {scores.mean():.4f} ± {scores.std():.4f}")

print("\n💡 'adaptive': divide lr por 5 cuando la pérdida deja de bajar (similar a ReduceLROnPlateau)")
print("   En práctica: Adam con lr=0.001 y early_stopping suele ser el punto de partida más robusto.")

---
## Sección 4 — Introducción a PyTorch: Tu Primera Red Personalizada

`sklearn` es excelente para prototipado rápido, pero para arquitecturas no estándar,
control total del training loop o investigación, se necesita PyTorch (o JAX/TF).

**Por qué PyTorch:**
- Define la arquitectura como código Python explícito
- Diferenciación automática (`autograd`) para calcular gradientes de cualquier cómputo
- Train loop explícito: total control sobre mini-batches, pérdida, checkpoints
- Ecosystem de investigación: la mayoría de papers publican código en PyTorch


In [ ]:
# ━━━ RED NEURONAL COMPLETA EN PYTORCH ━━━
# (Si PyTorch no está instalado, se usa sklearn como fallback)

if TORCH_AVAILABLE:
    # ── Preparar datos ──────────────────────────────────────────────────────
    from sklearn.preprocessing import LabelEncoder
    digits = load_digits()
    X_d, y_d = digits.data.astype(np.float32), digits.target

    X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
        X_d, y_d, test_size=0.2, random_state=RANDOM_STATE, stratify=y_d
    )
    scaler = StandardScaler()
    X_d_tr_sc = scaler.fit_transform(X_d_tr).astype(np.float32)
    X_d_te_sc  = scaler.transform(X_d_te).astype(np.float32)

    # Tensores PyTorch
    X_tr_t = torch.from_numpy(X_d_tr_sc)
    y_tr_t = torch.from_numpy(y_d_tr).long()
    X_te_t = torch.from_numpy(X_d_te_sc)
    y_te_t = torch.from_numpy(y_d_te).long()

    dataset_tr = TensorDataset(X_tr_t, y_tr_t)
    loader_tr  = DataLoader(dataset_tr, batch_size=32, shuffle=True)

    # ── Definir arquitectura ─────────────────────────────────────────────────
    class MLP_PyTorch(nn.Module):
        """MLP con Dropout y BatchNorm."""
        def __init__(self, n_in, n_hidden1, n_hidden2, n_out, dropout_rate=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_in, n_hidden1),
                nn.BatchNorm1d(n_hidden1),    # ← BatchNorm después de la capa lineal
                nn.ReLU(),
                nn.Dropout(dropout_rate),      # ← Dropout
                nn.Linear(n_hidden1, n_hidden2),
                nn.BatchNorm1d(n_hidden2),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(n_hidden2, n_out)
            )

        def forward(self, x):
            return self.net(x)   # CrossEntropyLoss espera logits (sin softmax)

    model = MLP_PyTorch(n_in=64, n_hidden1=128, n_hidden2=64, n_out=10, dropout_rate=0.3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight_decay = L2

    print(f"Arquitectura:")
    print(model)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nParámetros entrenables: {n_params:,}")
else:
    print("⚠️  PyTorch no disponible — usando sklearn como demostración equivalente")
    pipe_torch_equiv = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                              solver='adam', alpha=1e-4, max_iter=200,
                              random_state=RANDOM_STATE))
    ])

In [ ]:
# ━━━ TRAINING LOOP PYTORCH ━━━
if TORCH_AVAILABLE:
    N_EPOCHS = 50
    train_losses, val_accs = [], []

    for epoch in range(N_EPOCHS):
        # ── Fase de entrenamiento ──
        model.train()  # activa dropout y batch norm en modo train
        epoch_loss = 0.0
        for batch_X, batch_y in loader_tr:
            optimizer.zero_grad()       # 1. limpiar gradientes
            logits = model(batch_X)     # 2. forward pass
            loss = criterion(logits, batch_y)  # 3. calcular pérdida
            loss.backward()             # 4. backpropagation
            optimizer.step()            # 5. actualizar pesos
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(loader_tr))

        # ── Fase de evaluación ──
        model.eval()   # desactiva dropout y usa estadísticas acumuladas de batchnorm
        with torch.no_grad():
            logits_te = model(X_te_t)
            preds_te  = logits_te.argmax(dim=1).numpy()
            val_acc   = accuracy_score(y_d_te, preds_te)
        val_accs.append(val_acc)

        if (epoch + 1) % 10 == 0:
            print(f"Época {epoch+1:3d}/{N_EPOCHS} | Loss: {train_losses[-1]:.4f} | Val Acc: {val_acc:.4f}")

    # ── Gráficas ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, color='steelblue', linewidth=2)
    axes[0].set_title('Pérdida de Entrenamiento'); axes[0].set_xlabel('Época'); axes[0].set_ylabel('CE Loss')
    axes[1].plot(val_accs, color='tomato', linewidth=2)
    axes[1].set_title('Accuracy en Validación'); axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
    axes[1].axhline(y=max(val_accs), color='gray', linestyle='--', alpha=0.5,
                    label=f'Best: {max(val_accs):.4f}')
    axes[1].legend()
    plt.suptitle('Training Loop — MLP con BatchNorm + Dropout (PyTorch)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"\n✅ Accuracy final en test: {val_accs[-1]:.4f} | Mejor: {max(val_accs):.4f}")
else:
    digits = load_digits()
    pipe_torch_equiv.fit(digits.data[:1437], digits.target[:1437])
    acc = accuracy_score(digits.target[1437:], pipe_torch_equiv.predict(digits.data[1437:]))
    print(f"✅ Accuracy equivalente (sklearn): {acc:.4f}")

---
## Sección 5 — Experimento: Comparar Estrategias de Regularización (10 min)

¿Cuánto ayuda cada técnica? Comparamos sistemáticamente sobre el mismo dataset.


In [ ]:
# ━━━ EXPERIMENTO: L2 vs DROPOUT vs AMBOS ━━━
from sklearn.model_selection import cross_val_score

X_exp, y_exp = load_digits().data, load_digits().target

configs_reg = {
    'Sin regularización':      dict(alpha=0.0,    activation='relu'),
    'Solo L2 (alpha=0.01)':    dict(alpha=0.01,   activation='relu'),
    'Comparación ReLU vs tanh':dict(alpha=0.001,  activation='tanh'),
    'L2 + early_stopping':     dict(alpha=0.001,  activation='relu'),
}

print("Comparación de estrategias de regularización — Dígitos (5-fold CV)\n")
print(f"{'Configuración':<35} {'CV Acc':>8} {'Std':>7}")
print("-" * 55)

for name, kwargs in configs_reg.items():
    es = name == 'L2 + early_stopping'
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100, 50), solver='adam',
                              max_iter=500, random_state=RANDOM_STATE,
                              early_stopping=es, **kwargs))
    ])
    scores = cross_val_score(pipe, X_exp, y_exp, cv=5, scoring='accuracy')
    marker = " ←mejor" if scores.mean() == max(
        cross_val_score(Pipeline([('sc', StandardScaler()),
            ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), solver='adam',
                                  max_iter=500, random_state=RANDOM_STATE,
                                  early_stopping=es2, **kw))]),
            X_exp, y_exp, cv=5, scoring='accuracy').mean()
        for es2, kw in [(n=='L2 + early_stopping', k) for n,k in configs_reg.items()]
    ) else ""
    print(f"{name:<35} {scores.mean():>8.4f} {scores.std():>7.4f}")

print("\n💡 Early stopping + L2 + Adam suele ser el combo más robusto para empezar.")

---
## Sección 6 — Ejercicio en Clase (10 min)

### Parte A — Sin computador (5 min) 🖊️

Dada una red con Dropout(p=0.5) en una capa oculta de 4 neuronas:

**1.** Durante el entrenamiento, ¿cuántas neuronas se apagan en expectativa?

**2.** Durante la inferencia, si los pesos son $\mathbf{w} = [0.8, -0.3, 1.2, 0.5]$, ¿qué escala
se aplica para mantener la esperanza constante (inverted dropout)? ¿Cuál sería el vector de pesos efectivo?

**3.** Si la función de pérdida es $\mathcal{L} = 0.5$, ¿en qué dirección debería moverse
el learning rate según Adam vs. SGD simple para reducirla?

*Escribe tu respuesta aquí antes de ejecutar el código:*


In [ ]:
# ━━━ VERIFICACIÓN PARTE A ━━━
p = 0.5
n_neuronas = 4
w = np.array([0.8, -0.3, 1.2, 0.5])

print("Verificación — Dropout")
print(f"Neuronas apagadas en expectativa: {n_neuronas * p:.0f} de {n_neuronas}")
print(f"\nInverted Dropout: los pesos activos se escalan por 1/(1-p) = {1/(1-p):.2f}")
print(f"Vector de pesos efectivo (escala): {w * (1/(1-p))}")
print(f"\nDurante INFERENCIA: todas las neuronas activas, sin escala adicional")
print(f"→ E[salida_train] = E[salida_test] ✅")
print(f"\nSGD:  Δw = -lr * grad  (paso constante en magnitud)")
print(f"Adam: Δw = -lr * (m_hat / sqrt(v_hat))  (paso adaptado por parámetro)")

In [ ]:
# ━━━ EJERCICIO CÓDIGO: ALPHA vs FRONTERA DE DECISIÓN ━━━
X_m, y_m = make_moons(n_samples=200, noise=0.3, random_state=RANDOM_STATE)
X_mtr, X_mte, y_mtr, y_mte = train_test_split(X_m, y_m, test_size=0.3, random_state=RANDOM_STATE)

alphas_ej = [1e-5, 1e-3, 0.1, 1.0, 10.0]
fig, axes = plt.subplots(1, len(alphas_ej), figsize=(16, 3.5))

h = 0.02
xx_e, yy_e = np.meshgrid(
    np.arange(X_m[:,0].min()-0.3, X_m[:,0].max()+0.3, h),
    np.arange(X_m[:,1].min()-0.3, X_m[:,1].max()+0.3, h)
)

for ax, a in zip(axes, alphas_ej):
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), alpha=a,
                              max_iter=1000, random_state=RANDOM_STATE))
    ])
    pipe.fit(X_mtr, y_mtr)
    Z = pipe.predict(np.c_[xx_e.ravel(), yy_e.ravel()]).reshape(xx_e.shape)
    ax.contourf(xx_e, yy_e, Z, alpha=0.35, cmap='RdBu')
    ax.scatter(X_m[:,0], X_m[:,1], c=y_m, cmap='RdBu', edgecolors='black', s=20, alpha=0.7)
    te_acc = accuracy_score(y_mte, pipe.predict(X_mte))
    ax.set_title(f'α={a:.0e}\nAcc={te_acc:.3f}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Efecto de L2 (alpha) en la Frontera de Decisión — Moons', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Sección 7 — Resumen y Cierre de la Unidad

### 📊 Tabla de técnicas de regularización

| Técnica | Dónde actúa | Efecto | Hiperparámetro |
|---------|------------|--------|----------------|
| L2 (Weight Decay) | Función de pérdida | Penaliza pesos grandes | alpha |
| Dropout | Activaciones capas ocultas | Apaga neuronas aleatoriamente | dropout_rate (0.2–0.5) |
| Batch Normalization | Activaciones capas ocultas | Normaliza distribución interna | (sin HP principal) |
| Early Stopping | Loop de entrenamiento | Detiene cuando val_loss sube | patience |

### 📊 Tabla de optimizadores

| Optimizador | Cuándo usar | Notas |
|------------|------------|-------|
| Adam | Punto de partida universal | lr=0.001 casi siempre funciona |
| AdamW | Cuando L2 importa mucho | Desacopla weight decay del paso adaptativo |
| SGD + Momentum | Cuando se quiere mejor generalización final | Requiere más tuning de lr |
| RMSprop | RNNs y problemas no estacionarios | Precursor de Adam |

### 🔗 Cierre de la Unidad 3 — Redes Neuronales

Has completado la base de redes neuronales:

| Clase 1 | Clase 2 |
|---------|---------|
| Perceptrón → MLP → Backpropagation | Regularización → Optimización → PyTorch |

La **próxima unidad** aplica estos fundamentos a arquitecturas especializadas:
- **Redes Convolucionales (CNN)**: invarianza espacial para imágenes
- **Redes Recurrentes (RNN/LSTM)**: secuencias y series temporales
- **Transformers**: atención y modelos de lenguaje (el estado del arte actual)

---

### 📚 Bibliografía

#### Pregrado
- Géron, A. (2022). *Hands-On ML* (3ª ed.). Cap. 11 (Training Deep Networks).
- Srivastava, N. et al. (2014). Dropout. *JMLR* 15, 1929–1958.

#### Doctorado / Investigación
- Kingma, D. P., & Ba, J. (2015). Adam. *ICLR 2015*. arXiv:1412.6980
- Ioffe, S., & Szegedy, C. (2015). Batch Normalization. *ICML 2015*.
- Goodfellow et al. (2016). *Deep Learning*. MIT Press. Cap. 7 (Regularización) y 8 (Optimización).
- Wilson, A.C. et al. (2017). The marginal value of momentum for small learning rate SGD. *ICLR 2018*.
